# 🌿 EcoView: Student Model Distillation, ONNX Optimization & Quantization Pipeline

Welcome to the end-to-end training and optimization pipeline for the **EcoView** student classifier model. This notebook is designed to run in a standard Kaggle notebook environment (using a free GPU or CPU) to:

1. **Generate/Load Data**: Load VLM teacher-labeled images from the data flywheel (or generate a visual synthetic dataset if bootstrapping).
2. **Train student model**: Fine-tune a lightweight pre-trained `MobileNetV2` classifier head using transfer learning.
3. **ONNX Export**: Convert the trained PyTorch model weights to an optimized **FP32 ONNX** model.
4. **Quantize model**: Perform dynamic **INT8 Quantization** to reduce the model size by ~75% and boost CPU execution speed.
5. **Benchmark**: Perform detailed comparison tests between PyTorch, FP32 ONNX, and INT8 ONNX regarding **Storage Size**, **Inference Latency**, and **Classification Accuracy**.

---

### Architecture & Distillation Flow

Large Vision-Language Models (VLMs) like Gemini 2.5 Flash are highly accurate but too slow and costly for edge device deployment. In EcoView, we use a Teacher-Student paradigm to resolve this:

```mermaid
graph TD
    A[User Reports Image] -->|Gemini VLM Teacher| B(Auto-Labels Data)
    B -->|Saves to Flywheel| C[flywheel/ image + json]
    C -->|Distill via this notebook| D[PyTorch MobileNetV2]
    D -->|Export| E[FP32 ONNX Model]
    E -->|Quantize| F[INT8 Quantized ONNX]
    F -->|Deploy Edge ML service| G[Fast/Cheap Production Classifier]
```



In [ ]:
# Install required packages quietly to avoid messy output logs
!pip install -q onnxruntime onnxruntime-quantization torch torchvision pillow numpy scikit-learn matplotlib seaborn pandas tqdm


In [ ]:
import os
import sys
import json
import glob
import time
import random
from typing import List, Tuple
from PIL import Image, ImageDraw

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from sklearn.metrics import confusion_matrix, classification_report, f1_score, precision_recall_fscore_support

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

# ONNX runtime imports
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

# Set seaborn styling for premium charts
sns.set_theme(style="whitegrid")
plt.rcParams["figure.facecolor"] = "#ffffff"
plt.rcParams["axes.facecolor"] = "#f8f9fa"

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
print("[+] Libraries successfully imported!")
print("[+] PyTorch Version:", torch.__version__)
print("[+] ONNX Runtime Version:", ort.__version__)
print("[+] GPU (CUDA) Available:", torch.cuda.is_available())


In [ ]:
# Pipeline configurations and constants
CONFIG = {
    "labels": [
        "garbage_dump",
        "plastic_waste",
        "industrial_smoke",
        "water_contamination",
        "deforestation",
        "oil_spill",
    ],
    "image_size": (224, 224),
    "batch_size": 16,
    "epochs": 12,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "data_dir": "./data/flywheel",
    "output_dir": "./outputs",
    "seed": 42
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("[+] Configurations initialized. Target classification labels:")
for idx, label in enumerate(CONFIG["labels"]):
    print(f"  [{idx}] {label}")


In [ ]:
class EcoDataset(Dataset):
    def __init__(self, image_paths: List[str], label_indices: List[int], transform=None):
        self.image_paths = image_paths
        self.label_indices = label_indices
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.label_indices[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

def load_flywheel_dataset(data_dir: str) -> Tuple[List[str], List[int]]:
    """Reads real images and JSON annotations from the local flywheel data folder."""
    image_paths = []
    label_indices = []
    
    json_pattern = os.path.join(data_dir, "*.json")
    json_files = glob.glob(json_pattern)
    
    for json_file in json_files:
        try:
            with open(json_file, 'r') as f:
                data = json.load(f)
            category = data.get("category")
            if category in CONFIG["labels"]:
                label_idx = CONFIG["labels"].index(category)
                
                base_path = os.path.splitext(json_file)[0]
                img_path = None
                for ext in [".jpg", ".png", ".jpeg", ".JPG", ".PNG", ".JPEG"]:
                    test_path = base_path + ext
                    if os.path.exists(test_path):
                        img_path = test_path
                        break
                        
                if img_path:
                    image_paths.append(img_path)
                    label_indices.append(label_idx)
        except:
            pass
                
    return image_paths, label_indices

def generate_synthetic_image(label_name: str) -> Image.Image:
    """Generates clean, distinctive synthetic visual patterns for each class
    so the MobileNet model can learn features easily."""
    img = Image.new("RGB", (224, 224), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    
    if label_name == "garbage_dump":
        # Grey ground, brown/black trash piles
        img = Image.new("RGB", (224, 224), color=(195, 185, 175))
        draw = ImageDraw.Draw(img)
        for _ in range(12):
            x = random.randint(30, 190)
            y = random.randint(30, 190)
            r = random.randint(12, 28)
            draw.ellipse([x-r, y-r, x+r, y+r], fill=(105, 80, 55))
            
    elif label_name == "plastic_waste":
        # Sandy ground with small, colorful primary colored rectangles
        img = Image.new("RGB", (224, 224), color=(235, 215, 185))
        draw = ImageDraw.Draw(img)
        for _ in range(25):
            x = random.randint(15, 205)
            y = random.randint(15, 205)
            w = random.randint(5, 14)
            h = random.randint(5, 14)
            color = random.choice([
                (240, 40, 40),   # Red plastic
                (40, 90, 240),   # Blue plastic
                (240, 210, 40),  # Yellow plastic
                (40, 190, 80)    # Green plastic
            ])
            draw.rectangle([x, y, x+w, y+h], fill=color)
            
    elif label_name == "industrial_smoke":
        # Sky blue background with rising overlapping dark grey clouds
        img = Image.new("RGB", (224, 224), color=(170, 200, 235))
        draw = ImageDraw.Draw(img)
        for i in range(6):
            x = 85 + i * 12 + random.randint(-4, 4)
            y = 175 - i * 28 + random.randint(-8, 8)
            r = 16 + i * 7
            draw.ellipse([x-r, y-r, x+r, y+r], fill=(75, 75, 75))
        # Draw base factory chimney
        draw.rectangle([75, 165, 95, 224], fill=(45, 45, 45))
        
    elif label_name == "water_contamination":
        # Teal/deep blue water with toxic bright green algae patterns
        img = Image.new("RGB", (224, 224), color=(15, 60, 145))
        draw = ImageDraw.Draw(img)
        for _ in range(9):
            x = random.randint(25, 195)
            y = random.randint(25, 195)
            r = random.randint(18, 40)
            draw.ellipse([x-r, y-r, x+r, y+r], fill=(0, 220, 110))
            
    elif label_name == "deforestation":
        # Earthy background, green patch rings and brown/orange cut tree circles
        img = Image.new("RGB", (224, 224), color=(150, 110, 70))
        draw = ImageDraw.Draw(img)
        for _ in range(6):
            x = random.randint(30, 190)
            y = random.randint(30, 190)
            r = random.randint(25, 45)
            draw.ellipse([x-r, y-r, x+r, y+r], fill=(30, 130, 30))
        for _ in range(15):
            x = random.randint(15, 205)
            y = random.randint(15, 205)
            draw.rectangle([x-4, y-4, x+4, y+4], fill=(75, 45, 20))
            
    elif label_name == "oil_spill":
        # Deep blue sea with large overlapping jet black slick rings
        img = Image.new("RGB", (224, 224), color=(25, 65, 115))
        draw = ImageDraw.Draw(img)
        x, y, r = 112, 112, 55
        draw.ellipse([x-r, y-r, x+r, y+r], fill=(5, 5, 5))
        for _ in range(5):
            ox = random.randint(70, 150)
            oy = random.randint(70, 150)
            or_ = random.randint(20, 40)
            draw.ellipse([ox-or_, oy-or_, ox+or_, oy+or_], fill=(5, 5, 5))
            
    return img

def get_or_create_dataset(data_dir: str, num_synthetic_per_class: int = 40) -> Tuple[List[str], List[int], List[str], List[int]]:
    # Try to load existing data flywheel
    real_paths, real_labels = load_flywheel_dataset(data_dir)
    
    if len(real_paths) >= 12:
        print(f"[+] Loaded {len(real_paths)} real samples from EcoView flywheel.")
        # Splitting dataset
        indices = list(range(len(real_paths)))
        random.shuffle(indices)
        split = int(len(real_paths) * 0.8)
        
        return ([real_paths[i] for i in indices[:split]], [real_labels[i] for i in indices[:split]],
                [real_paths[i] for i in indices[split:]], [real_labels[i] for i in indices[split:]])
                
    print("[!] Insufficient local flywheel data. Bootstrapping synthetic dataset for Kaggle training...")
    os.makedirs(os.path.join(data_dir, "train"), exist_ok=True)
    os.makedirs(os.path.join(data_dir, "val"), exist_ok=True)
    
    train_paths, train_labels = [], []
    val_paths, val_labels = [], []
    
    for label_idx, label in enumerate(CONFIG["labels"]):
        # Train images
        for i in range(num_synthetic_per_class):
            img = generate_synthetic_image(label)
            path = os.path.join(data_dir, "train", f"{label}_{i}.jpg")
            img.save(path)
            # Create a corresponding label file
            meta = {"category": label, "is_pollution": True, "confidence": 0.95}
            with open(path.replace(".jpg", ".json"), "w") as f:
                json.dump(meta, f)
            train_paths.append(path)
            train_labels.append(label_idx)
            
        # Validation images
        for i in range(15):
            img = generate_synthetic_image(label)
            path = os.path.join(data_dir, "val", f"{label}_{i}.jpg")
            img.save(path)
            meta = {"category": label, "is_pollution": True, "confidence": 0.95}
            with open(path.replace(".jpg", ".json"), "w") as f:
                json.dump(meta, f)
            val_paths.append(path)
            val_labels.append(label_idx)
            
    print(f"[+] Success: Generated {len(train_paths)} training samples and {len(val_paths)} validation samples.")
    return train_paths, train_labels, val_paths, val_labels

train_img, train_lbl, val_img, val_lbl = get_or_create_dataset(CONFIG["data_dir"])

# Visualize the dataset splits
df_train = pd.DataFrame({"class": [CONFIG["labels"][i] for i in train_lbl], "split": "Train"})
df_val = pd.DataFrame({"class": [CONFIG["labels"][i] for i in val_lbl], "split": "Validation"})
df_all = pd.concat([df_train, df_val])

plt.figure(figsize=(10, 4.5))
ax = sns.countplot(data=df_all, y="class", hue="split", palette=["#4f46e5", "#ec4899"], edgecolor="black")
plt.title("EcoView Dataset Distribution", fontsize=14, weight='bold', pad=15)
plt.xlabel("Sample Count", fontsize=12)
plt.ylabel("Classification Category", fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.legend(frameon=True, facecolor='#f8f9fa')
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/class_distribution.png", dpi=150)
plt.show()


In [ ]:
# Define transforms - keeping colors intact (no excessive color modifications)
train_transform = transforms.Compose([
    transforms.Resize(CONFIG["image_size"]),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(CONFIG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Visualize visual characteristics of each class
fig, axes = plt.subplots(2, 3, figsize=(14, 9.5))
axes = axes.flatten()

for idx, label_name in enumerate(CONFIG["labels"]):
    sample_idx = train_lbl.index(idx)
    img = Image.open(train_img[sample_idx])
    
    axes[idx].imshow(img)
    axes[idx].set_title(f"{label_name.replace('_', ' ').title()}", fontsize=12, weight='bold', pad=10)
    axes[idx].axis('off')
    
plt.suptitle("Sample Class Visual Patterns (Dataset Previews)", fontsize=16, weight='bold', y=0.98)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/sample_dataset_grid.png", dpi=150)
plt.show()


In [ ]:
print("[*] Instantiating pre-trained MobileNetV2 student network...")
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Freeze backbone layers
for param in model.features.parameters():
    param.requires_grad = False
    
# Replace standard classification head
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(in_features, len(CONFIG["labels"]))
)

# Parameters statistics
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\n[+] Student Architecture successfully defined:")
print(f"  - Total Parameters:     {total_params:,}")
print(f"  - Trainable Parameters: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")
print(f"  - Frozen Parameters:    {frozen_params:,} ({frozen_params/total_params*100:.2f}%)")


In [ ]:
train_dataset = EcoDataset(train_img, train_lbl, transform=train_transform)
val_dataset = EcoDataset(val_img, val_lbl, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print(f"[*] Commencing training loop on device: {device}...")
for epoch in range(CONFIG["epochs"]):
    # Training step
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    
    # Validation step
    model.eval()
    val_loss_running, val_correct, val_total = 0.0, 0, 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss_running += loss.item() * inputs.size(0)
            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()
            
    epoch_val_loss = val_loss_running / val_total
    epoch_val_acc = val_correct / val_total
    
    history["train_loss"].append(epoch_loss)
    history["train_acc"].append(epoch_acc)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(epoch_val_acc)
    
    print(f"Epoch [{epoch+1:02d}/{CONFIG['epochs']:02d}] "
          f"Train Loss: {epoch_loss:.4f} - Train Acc: {epoch_acc*100:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc*100:.2f}%")

pytorch_path = f"{CONFIG['output_dir']}/student_model.pth"
torch.save(model.state_dict(), pytorch_path)
print(f"\n[+] Saved trained PyTorch weights to: {pytorch_path}")

# Plot learning histories
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

ax1.plot(range(1, CONFIG["epochs"]+1), history["train_loss"], 'o-', label="Train Loss", color="#4f46e5", linewidth=2.5)
ax1.plot(range(1, CONFIG["epochs"]+1), history["val_loss"], 's-', label="Val Loss", color="#db2777", linewidth=2.5)
ax1.set_title("Epoch Loss Trends", fontsize=13, weight='bold', pad=10)
ax1.set_xlabel("Epoch Index")
ax1.set_ylabel("Loss Magnitude")
ax1.legend(frameon=True)
ax1.grid(True, alpha=0.4)

ax2.plot(range(1, CONFIG["epochs"]+1), [a*100 for a in history["train_acc"]], 'o-', label="Train Acc", color="#4f46e5", linewidth=2.5)
ax2.plot(range(1, CONFIG["epochs"]+1), [a*100 for a in history["val_acc"]], 's-', label="Val Acc", color="#db2777", linewidth=2.5)
ax2.set_title("Epoch Accuracy Trends", fontsize=13, weight='bold', pad=10)
ax2.set_xlabel("Epoch Index")
ax2.set_ylabel("Accuracy (%)")
ax2.legend(frameon=True)
ax2.grid(True, alpha=0.4)

plt.suptitle("EcoView Student Training Diagnostic Curves", fontsize=16, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/training_history.png", dpi=150)
plt.show()


In [ ]:
print("[*] Exporting MobileNetV2 student model to ONNX format (FP32)...")
model.eval()
model.to("cpu")  # Dynamic conversion matches CPU usage profiles

dummy_input = torch.randn(1, 3, 224, 224)
onnx_path = f"{CONFIG['output_dir']}/student_model.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

if os.path.exists(onnx_path):
    fp32_size = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"[+] FP32 ONNX model successfully saved to: {onnx_path}")
    print(f"  - Output Size: {fp32_size:.2f} MB")
else:
    print("[!] ONNX Export failure detected.")


In [ ]:
print("[*] Applying Dynamic INT8 Quantization...")
quantized_path = f"{CONFIG['output_dir']}/student_model_quantized.onnx"

try:
    quantize_dynamic(
        model_input=onnx_path,
        model_output=quantized_path,
        weight_type=QuantType.QUInt8
    )
    
    if os.path.exists(quantized_path):
        int8_size = os.path.getsize(quantized_path) / (1024 * 1024)
        print(f"[+] Quantized INT8 ONNX model successfully saved to: {quantized_path}")
        print(f"  - Output Size: {int8_size:.2f} MB")
    else:
        print("[!] Dynamic quantization output file not found.")
except Exception as e:
    print(f"[!] Quantization encountered an exception: {e}")

# Compare weights storage sizes on disk
sizes = {
    "PyTorch Weights (.pth)": os.path.getsize(pytorch_path) / (1024*1024),
    "ONNX FP32 (.onnx)": os.path.getsize(onnx_path) / (1024*1024),
    "ONNX INT8 (.onnx quantized)": os.path.getsize(quantized_path) / (1024*1024)
}

plt.figure(figsize=(9, 4.5))
bars = plt.barh(list(sizes.keys()), list(sizes.values()), color=["#a855f7", "#6366f1", "#06b6d4"], edgecolor="black")
plt.title("Model Storage Requirements Comparison (Lower is Better)", fontsize=13, weight='bold', pad=15)
plt.xlabel("File Size on Disk (MB)", fontsize=11)
plt.grid(axis='x', linestyle='--', alpha=0.5)

# Value annotations on chart
for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.15, bar.get_y() + bar.get_height()/2, f"{w:.2f} MB", 
             va='center', ha='left', weight='bold')

plt.xlim(0, max(sizes.values()) * 1.15)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/model_size_comparison.png", dpi=150)
plt.show()

compression_ratio = (1 - (sizes["ONNX INT8 (.onnx quantized)"] / sizes["ONNX FP32 (.onnx)"])) * 100
print(f"[+] Edge compression: achieved {compression_ratio:.1f}% reduction in final footprint!")


In [ ]:
print("[*] Commencing execution latency benchmarking...")

# Run Inference sessions using CPUExecutionProvider
ort_sess_fp32 = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
ort_sess_int8 = ort.InferenceSession(quantized_path, providers=['CPUExecutionProvider'])

dummy_np = np.random.randn(1, 3, 224, 224).astype(np.float32)
dummy_torch = torch.tensor(dummy_np)

# Warmups
_ = model(dummy_torch)
_ = ort_sess_fp32.run(None, {'input': dummy_np})
_ = ort_sess_int8.run(None, {'input': dummy_np})

num_runs = 100

# 1. PyTorch CPU
t_start = time.perf_counter()
model.to("cpu")
model.eval()
with torch.no_grad():
    for _ in range(num_runs):
        _ = model(dummy_torch)
t_end = time.perf_counter()
pytorch_cpu_ms = ((t_end - t_start) / num_runs) * 1000

# 2. PyTorch GPU (if GPU is available)
pytorch_gpu_ms = None
if torch.cuda.is_available():
    model.to(device)
    dummy_gpu = dummy_torch.to(device)
    # Warmup
    _ = model(dummy_gpu)
    t_start = time.perf_counter()
    with torch.no_grad():
        for _ in range(num_runs):
            _ = model(dummy_gpu)
            torch.cuda.synchronize()
    t_end = time.perf_counter()
    pytorch_gpu_ms = ((t_end - t_start) / num_runs) * 1000
    model.to("cpu")

# 3. ONNX FP32 CPU
t_start = time.perf_counter()
for _ in range(num_runs):
    _ = ort_sess_fp32.run(None, {'input': dummy_np})
t_end = time.perf_counter()
onnx_fp32_ms = ((t_end - t_start) / num_runs) * 1000

# 4. ONNX INT8 CPU
t_start = time.perf_counter()
for _ in range(num_runs):
    _ = ort_sess_int8.run(None, {'input': dummy_np})
t_end = time.perf_counter()
onnx_int8_ms = ((t_end - t_start) / num_runs) * 1000

# Plot Latency Comparison
latencies = {
    "PyTorch (CPU)": pytorch_cpu_ms,
    "ONNX FP32 (CPU)": onnx_fp32_ms,
    "ONNX INT8 (CPU)": onnx_int8_ms
}
if pytorch_gpu_ms:
    latencies["PyTorch (GPU)"] = pytorch_gpu_ms

latencies = dict(sorted(latencies.items(), key=lambda x: x[1], reverse=True))

plt.figure(figsize=(9, 4.5))
bars = plt.barh(list(latencies.keys()), list(latencies.values()), color=["#f43f5e", "#d946ef", "#8b5cf6", "#06b6d4"], edgecolor="black")
plt.title("Model Inference Latency (Lower is Better)", fontsize=13, weight='bold', pad=15)
plt.xlabel("Average Time per Image (ms)", fontsize=11)
plt.grid(axis='x', linestyle='--', alpha=0.5)

for bar in bars:
    w = bar.get_width()
    plt.text(w + (max(latencies.values()) * 0.015), bar.get_y() + bar.get_height()/2, f"{w:.2f} ms", 
             va='center', ha='left', weight='bold')

plt.xlim(0, max(latencies.values()) * 1.15)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/model_latency_comparison.png", dpi=150)
plt.show()

speedup = pytorch_cpu_ms / onnx_int8_ms
print(f"[+] Performance optimization: INT8 ONNX is {speedup:.1f}x faster than PyTorch CPU!")

# Accuracy metrics on validation set
print("\n[*] Evaluating validation set predictions on quantized model...")
y_true, y_pred = [], []
for idx in range(len(val_img)):
    img = Image.open(val_img[idx]).convert("RGB")
    tensor = val_transform(img).unsqueeze(0).numpy()
    
    outputs = ort_sess_int8.run(None, {'input': tensor})[0]
    pred = np.argmax(outputs, axis=1)[0]
    
    y_true.append(val_lbl[idx])
    y_pred.append(pred)

accuracy = np.sum(np.array(y_true) == np.array(y_pred)) / len(y_true)
print(f"[+] INT8 Quantized Model Accuracy: {accuracy*100:.2f}%")

# Generate and plot confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8.5, 7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CONFIG["labels"], yticklabels=CONFIG["labels"], cmap="Purples", cbar=True)
plt.title("Confusion Matrix (ONNX INT8 quantized)", fontsize=13, weight='bold', pad=15)
plt.xlabel("Predicted Category", fontsize=11)
plt.ylabel("True Category", fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/confusion_matrix.png", dpi=150)
plt.show()

# Show Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CONFIG["labels"]))

# Plot per-class precision, recall, and F1
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average=None)
df_metrics = pd.DataFrame({
    "Category": [c.replace('_', ' ').title() for c in CONFIG["labels"]] * 3,
    "Score": list(prec) + list(rec) + list(f1),
    "Metric": ["Precision"]*6 + ["Recall"]*6 + ["F1-Score"]*6
})

plt.figure(figsize=(11, 6.5))
sns.barplot(data=df_metrics, y="Category", x="Score", hue="Metric", palette="pastel", edgecolor="black")
plt.title("Per-Class Metric Profiles (ONNX INT8 quantized)", fontsize=13, weight='bold', pad=15)
plt.xlim(0, 1.1)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.legend(frameon=True, facecolor='#f8f9fa')
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/class_metrics.png", dpi=150)
plt.show()


In [ ]:
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=-1)

# Visualize inference on 3 random samples from validation set
sample_indices = random.sample(range(len(val_img)), 3)

for idx in sample_indices:
    img_path = val_img[idx]
    true_label = CONFIG["labels"][val_lbl[idx]]
    
    img = Image.open(img_path).convert("RGB")
    tensor = val_transform(img).unsqueeze(0).numpy()
    
    logits = ort_sess_int8.run(None, {'input': tensor})[0][0]
    probs = softmax(logits)
    pred_idx = np.argmax(probs)
    pred_label = CONFIG["labels"][pred_idx]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    
    # Plot Image
    ax1.imshow(img)
    ax1.set_title(f"True: {true_label.replace('_', ' ').title()}\nPred: {pred_label.replace('_', ' ').title()} ({probs[pred_idx]*100:.1f}%)", 
                  color="green" if true_label == pred_label else "red", weight='bold', fontsize=12)
    ax1.axis('off')
    
    # Plot Probabilities
    colors = ["#3b82f6" if label == pred_label else "#cbd5e1" for label in CONFIG["labels"]]
    y_pos = np.arange(len(CONFIG["labels"]))
    labels_clean = [l.replace('_', ' ').title() for l in CONFIG["labels"]]
    ax2.barh(y_pos, probs, color=colors, edgecolor="black")
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(labels_clean, fontsize=10)
    ax2.invert_yaxis()
    ax2.set_xlabel("Probability Class Confidence", fontsize=11)
    ax2.set_xlim(0, 1.1)
    ax2.grid(axis='x', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()


In [ ]:
summary_df = pd.DataFrame([
    {
        "Model Configuration": "PyTorch (CPU)",
        "Storage footprint": f"{sizes['PyTorch Weights (.pth)']:.2f} MB",
        "Avg Inference Latency": f"{pytorch_cpu_ms:.2f} ms",
        "Accuracy Score": "N/A"
    },
    {
        "Model Configuration": "ONNX FP32 (CPU)",
        "Storage footprint": f"{sizes['ONNX FP32 (.onnx)']:.2f} MB",
        "Avg Inference Latency": f"{onnx_fp32_ms:.2f} ms",
        "Accuracy Score": f"{accuracy*100:.2f}%"
    },
    {
        "Model Configuration": "ONNX INT8 (CPU)",
        "Storage footprint": f"{sizes['ONNX INT8 (.onnx quantized)']:.2f} MB",
        "Avg Inference Latency": f"{onnx_int8_ms:.2f} ms",
        "Accuracy Score": f"{accuracy*100:.2f}%"
    }
])

if pytorch_gpu_ms:
    gpu_row = pd.DataFrame([{
        "Model Configuration": "PyTorch (GPU)",
        "Storage footprint": f"{sizes['PyTorch Weights (.pth)']:.2f} MB",
        "Avg Inference Latency": f"{pytorch_gpu_ms:.2f} ms",
        "Accuracy Score": "N/A"
    }])
    summary_df = pd.concat([summary_df, gpu_row], ignore_index=True)

# Render HTML table representation
import IPython.display as display
display.display(display.HTML(summary_df.to_html(index=False, classes="table table-bordered table-striped")))


## 🚀 Deployment Instructions for the Quantized Model

Now that the student model has been distilled and optimized, follow these steps to integrate the model back into your EcoView workspace:

### 1. Download the Quantized Model
* Expand the **Output** section in the right sidebar of the Kaggle notebook UI.
* Navigate inside `/kaggle/working/outputs/` and locate the file named `student_model_quantized.onnx`.
* Click the download icon next to it and save it to your local machine.

### 2. Copy the Model to the ML Backend Workspace
* Move the downloaded `student_model_quantized.onnx` file to the root of your Python backend:
  `d:\synced-pc\1_Work\projects\Ecoview\EcoView\ecoview-ml-backend\student_model_quantized.onnx`

### 3. Deploy the Microservice
* When you launch the ML backend using `docker` or `uvicorn`, it checks for the existence of this quantized model at its root path:
  * If found, the FastAPI backend boots using **ONNX Runtime CPU** to classify incoming reports within 30 milliseconds.
  * If missing, the backend defaults to the external **Gemini 2.5 Flash VLM** validator, incurring minor latency and REST API costs.

